# Lab 3.3: Path-Specific Rules with Glob Patterns

**What you'll build:** A rules configuration system using .claude/rules/ with YAML frontmatter and glob patterns -- and learn why glob-scoped rules beat duplicated directory CLAUDE.md files for cross-cutting concerns.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- duplicated CLAUDE.md files across test directories | 5 min |
| 2 | The Right Way -- single .claude/rules/ file with glob pattern | 5 min |
| 3 | Your Turn -- design rules files for a real project with multiple file types | 10 min |
| 4 | Stress Test -- validate your glob patterns match the right files | 5 min |

In [ ]:
import anthropic
import json
import fnmatch
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You are configuring Claude Code for a monorepo with test files in many directories:

```
src/
  api/
    users.ts
    users.test.ts
    orders.ts
    orders.test.ts
  auth/
    login.ts
    login.test.ts
    login.spec.ts
  billing/
    invoice.ts
    invoice.test.ts
  database/
    migrations/
      001_users.sql
      002_orders.sql
    seeds/
      users.sql
      orders.sql
```

All test files should follow the same conventions, regardless of which directory they are in.

In [ ]:
# Simulated file tree
PROJECT_FILES = [
    "src/api/users.ts",
    "src/api/users.test.ts",
    "src/api/orders.ts",
    "src/api/orders.test.ts",
    "src/auth/login.ts",
    "src/auth/login.test.ts",
    "src/auth/login.spec.ts",
    "src/billing/invoice.ts",
    "src/billing/invoice.test.ts",
    "src/database/migrations/001_users.sql",
    "src/database/migrations/002_orders.sql",
    "src/database/seeds/users.sql",
    "src/database/seeds/orders.sql",
    "package.json",
    "tsconfig.json",
    "README.md",
]

# The WRONG way: duplicated CLAUDE.md in every test directory
WRONG_RULES = {
    "approach": "Directory CLAUDE.md in each test directory",
    "files": {
        "src/api/CLAUDE.md": "# Testing\n- Use describe/it blocks\n- Mock external services\n- One assertion per test",
        "src/auth/CLAUDE.md": "# Testing\n- Use describe/it blocks\n- Mock external services\n- One assertion per test",
        "src/billing/CLAUDE.md": "# Testing\n- Use describe/it blocks\n- Mock external services\n- One assertion per test",
    },
    "problems": [
        "3 copies of identical rules -- maintenance nightmare",
        "New directories need manual CLAUDE.md creation",
        "Rules apply to ALL files in directory, not just tests",
        "src/api/users.ts (not a test) also gets testing rules"
    ]
}

print("=== WRONG APPROACH: Duplicated CLAUDE.md files ===")
print(f"Files created: {len(WRONG_RULES['files'])} identical copies")
for path in WRONG_RULES['files']:
    print(f"  {path}")
print(f"\nProblems:")
for p in WRONG_RULES['problems']:
    print(f"  - {p}")

---

## Phase 1: The Wrong Approach

Let's demonstrate the problem: directory CLAUDE.md applies testing rules to non-test files.

In [ ]:
# Simulate which files get testing rules with directory CLAUDE.md
print("=== Files receiving testing rules (directory CLAUDE.md) ===")
dirs_with_rules = ["src/api/", "src/auth/", "src/billing/"]

correct_targets = []  # Test files that should get rules
false_targets = []    # Non-test files that incorrectly get rules
missed = []           # Test files without rules

for f in PROJECT_FILES:
    is_test = '.test.' in f or '.spec.' in f
    in_ruled_dir = any(f.startswith(d) for d in dirs_with_rules)
    
    if in_ruled_dir and is_test:
        correct_targets.append(f)
    elif in_ruled_dir and not is_test:
        false_targets.append(f)
    elif not in_ruled_dir and is_test:
        missed.append(f)

print(f"\nCorrectly targeted (test files with rules): {len(correct_targets)}")
for f in correct_targets:
    print(f"  [OK] {f}")

print(f"\nFalse targets (non-test files getting test rules): {len(false_targets)}")
for f in false_targets:
    print(f"  [WRONG] {f}")

print(f"\nMissed (test files without rules): {len(missed)}")
for f in missed:
    print(f"  [MISSED] {f}")

precision = len(correct_targets) / max(len(correct_targets) + len(false_targets), 1)
recall = len(correct_targets) / max(len(correct_targets) + len(missed), 1)
print(f"\nPrecision: {precision:.0%} (of files getting rules, how many are actually tests)")
print(f"Recall: {recall:.0%} (of test files, how many get the rules)")

---

## Phase 2: The Right Approach

A single `.claude/rules/testing.md` with glob patterns targets exactly the right files.

In [ ]:
# The right way: .claude/rules/ with glob patterns
RIGHT_RULES = {
    ".claude/rules/testing.md": {
        "frontmatter": {
            "paths": ["**/*.test.ts", "**/*.spec.ts"]
        },
        "content": """# Testing Standards
- Use describe/it blocks, not test()
- Mock all external services (API calls, database, file system)
- One assertion per test
- Name tests: test_<behavior>_<condition>_<expected>
- Test categories: happy path, edge cases, error cases"""
    }
}

def match_glob_patterns(filepath, patterns):
    """Check if a filepath matches any of the glob patterns."""
    for pattern in patterns:
        # Simple glob matching -- ** matches any depth
        if fnmatch.fnmatch(filepath, pattern):
            return True
        # Handle ** for recursive matching
        if '**' in pattern:
            simple_pattern = pattern.replace('**/', '')
            if fnmatch.fnmatch(filepath.split('/')[-1], simple_pattern):
                return True
    return False

# Test which files match the glob patterns
rule = RIGHT_RULES[".claude/rules/testing.md"]
patterns = rule["frontmatter"]["paths"]

print(f"=== Glob Pattern Matching ===")
print(f"Patterns: {patterns}")
print()

matched = []
unmatched = []
for f in PROJECT_FILES:
    if match_glob_patterns(f, patterns):
        matched.append(f)
    else:
        unmatched.append(f)

print(f"Matched ({len(matched)} files):")
for f in matched:
    is_test = '.test.' in f or '.spec.' in f
    tag = "CORRECT" if is_test else "FALSE MATCH"
    print(f"  [{tag}] {f}")

print(f"\nNot matched ({len(unmatched)} files):")
for f in unmatched:
    is_test = '.test.' in f or '.spec.' in f
    tag = "MISSED" if is_test else "CORRECT"
    print(f"  [{tag}] {f}")

# Calculate precision and recall
glob_tp = sum(1 for f in matched if '.test.' in f or '.spec.' in f)
glob_fp = sum(1 for f in matched if '.test.' not in f and '.spec.' not in f)
glob_fn = sum(1 for f in unmatched if '.test.' in f or '.spec.' in f)

glob_precision = glob_tp / max(glob_tp + glob_fp, 1)
glob_recall = glob_tp / max(glob_tp + glob_fn, 1)
print(f"\nPrecision: {glob_precision:.0%}")
print(f"Recall: {glob_recall:.0%}")

print(f"\n=== COMPARISON ===")
print(f"{'Metric':<20} {'Dir CLAUDE.md':>15} {'Glob Rules':>15}")
print(f"{'Precision':<20} {precision:>14.0%} {glob_precision:>14.0%}")
print(f"{'Recall':<20} {recall:>14.0%} {glob_recall:>14.0%}")
print(f"{'Files to maintain':<20} {len(WRONG_RULES['files']):>15} {'1':>15}")

---

## Phase 3: Your Turn

Design a complete set of `.claude/rules/` files for this project. You need rules for:
1. **Test files** (*.test.ts, *.spec.ts) -- testing conventions
2. **Migration SQL files** (migrations/**/*.sql) -- strict schema rules
3. **Seed SQL files** (seeds/**/*.sql) -- different, more relaxed rules
4. **TypeScript source files** (src/**/*.ts, excluding tests) -- coding standards
5. **Config files** (*.json at root) -- change management rules

In [ ]:
# =============================================================
# TODO: Design your .claude/rules/ configuration
# =============================================================

YOUR_RULES = {
    # TODO: Add your rule files
    # ".claude/rules/testing.md": {
    #     "frontmatter": {"paths": ["**/*.test.ts", "**/*.spec.ts"]},
    #     "content": "..."
    # },
    # ".claude/rules/migrations.md": {
    #     "frontmatter": {"paths": ["**/migrations/**/*.sql"]},
    #     "content": "..."
    # },
}

print(f"Your rules ({len(YOUR_RULES)} files):")
for name, rule in YOUR_RULES.items():
    print(f"  {name}")
    print(f"    paths: {rule.get('frontmatter', {}).get('paths', [])}")

In [ ]:
# =============================================================
# Checker: validates your rules design
# =============================================================

def check_rules(rules, project_files):
    errors = []
    
    # Check 1: Must have rules files
    if not rules:
        errors.append("No rules files defined.")
        print("=== RULES VALIDATION ===")
        print(f"  [ERROR] {errors[0]}")
        return False
    
    # Check 2: All rule files must have frontmatter with paths
    for name, rule in rules.items():
        fm = rule.get('frontmatter', {})
        if 'paths' not in fm or not fm['paths']:
            errors.append(f"{name}: missing 'paths' in frontmatter.")
        if not rule.get('content', '').strip():
            errors.append(f"{name}: empty content.")
    
    # Check 3: Test files should be covered
    test_files = [f for f in project_files if '.test.' in f or '.spec.' in f]
    test_covered = False
    for name, rule in rules.items():
        patterns = rule.get('frontmatter', {}).get('paths', [])
        for tf in test_files:
            if match_glob_patterns(tf, patterns):
                test_covered = True
                break
    if not test_covered:
        errors.append("No rule covers test files (*.test.ts, *.spec.ts).")
    
    # Check 4: Migration and seed SQL should have DIFFERENT rules
    migration_patterns = []
    seed_patterns = []
    for name, rule in rules.items():
        patterns = rule.get('frontmatter', {}).get('paths', [])
        for p in patterns:
            if 'migration' in p.lower():
                migration_patterns.append(p)
            if 'seed' in p.lower():
                seed_patterns.append(p)
    
    if not migration_patterns:
        errors.append("No rule specifically targets migration SQL files.")
    if not seed_patterns:
        errors.append("No rule specifically targets seed SQL files.")
    
    # Check 5: No overlapping patterns that would cause conflicts
    all_patterns = {}
    for name, rule in rules.items():
        for p in rule.get('frontmatter', {}).get('paths', []):
            if p in all_patterns:
                errors.append(f"Pattern '{p}' used in both {all_patterns[p]} and {name}.")
            all_patterns[p] = name
    
    # Results
    print("=== RULES VALIDATION ===")
    if not errors:
        print("PASSED -- Well-designed rules system!")
        print(f"  Rules files: {len(rules)}")
        print(f"  Total patterns: {len(all_patterns)}")
    else:
        for e in errors:
            print(f"  [ERROR] {e}")
        print(f"\nFix {len(errors)} error(s) and try again.")
    
    return len(errors) == 0

check_rules(YOUR_RULES, PROJECT_FILES)

---

## Phase 4: Stress Test

Let's test your glob patterns against the full file tree to verify precision and recall.

In [ ]:
# Stress test: verify each file gets exactly the right rules
print("=== FILE-BY-FILE RULE MATCHING ===")
print(f"{'File':<45} {'Rules Applied':>30}")
print("-" * 75)

for f in PROJECT_FILES:
    matching_rules = []
    for rule_name, rule in YOUR_RULES.items():
        patterns = rule.get('frontmatter', {}).get('paths', [])
        if match_glob_patterns(f, patterns):
            # Extract short name from rule file path
            short_name = rule_name.split('/')[-1].replace('.md', '')
            matching_rules.append(short_name)
    
    rules_str = ', '.join(matching_rules) if matching_rules else '(none)'
    print(f"  {f:<43} {rules_str:>30}")

# Additional test: verify no file gets conflicting rules
print("\n=== CONFLICT CHECK ===")
conflicts_found = False
for f in PROJECT_FILES:
    matching_rules = []
    for rule_name, rule in YOUR_RULES.items():
        patterns = rule.get('frontmatter', {}).get('paths', [])
        if match_glob_patterns(f, patterns):
            matching_rules.append(rule_name)
    
    if len(matching_rules) > 1:
        # Multiple rules is OK if they cover different concerns
        print(f"  [INFO] {f} matches {len(matching_rules)} rules: {matching_rules}")
        print(f"         (Multiple rules are OK if they cover different concerns)")

if not conflicts_found:
    print("  No problematic conflicts detected.")

### Key Takeaways

1. **`.claude/rules/` with glob patterns** handles cross-cutting concerns with a single file instead of duplicated CLAUDE.md files.
2. **`*` matches within one directory level, `**` matches across directory boundaries.** Use `**/*.test.ts` for files at any depth.
3. **Different file types in the same directory can have different rules.** Migration SQL and seed SQL get separate rule files.
4. **Rules files and directory CLAUDE.md both load and merge.** Use rules for file-type patterns, directory CLAUDE.md for location-specific context.
5. **YAML frontmatter with `paths:`** is the activation mechanism. No paths = rule never loads.